# Part (c) — multi-year simulated paths (3 calendar years)

Standalone notebook: **`Part(c).ipynb` is unchanged.** This repeats the minimal data + GARCH setup, then runs the batch that checks each month’s **simulated daily mean Close** against **`monthly_avg_early`** over a 3-year slice.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from partc import build_part_c_split
from partb import load_spx_daily, fit_t_garch_on_daily_recent
from partc_sim import (
    partc2_month_table,
    partc_feasible_calendar_year_starts,
    plot_partc_year_span_validation,
    simulate_partc2_year_span_batch,
)

monthly_avg_early, daily_recent, _ = build_part_c_split()
daily_full = load_spx_daily()
garch = fit_t_garch_on_daily_recent(daily_recent)


### Random 3-year window (set `seed` for reproducibility)

Use `start_year=2001` and `randomize_start_year=False` for a fixed window.


In [ ]:
mt_full_check = partc2_month_table(monthly_avg_early, daily_full)
print(
    "example feasible start years:",
    partc_feasible_calendar_year_starts(mt_full_check, n_years=3)[:12],
    "...",
)

YEAR_SPAN_BATCH = simulate_partc2_year_span_batch(
    monthly_avg_early,
    daily_full,
    garch,
    start_year=None,
    n_years=3,
    n_paths=20,
    seed=123,
    randomize_start_year=True,
    innovations="parametric",
)
print(
    "start_year",
    YEAR_SPAN_BATCH["start_year"],
    "n slice months",
    len(YEAR_SPAN_BATCH["month_table_slice"]),
    "all_ok",
    YEAR_SPAN_BATCH["all_ok"],
    "max_abs_error",
    YEAR_SPAN_BATCH["max_abs_error"],
)
assert YEAR_SPAN_BATCH["all_ok"], "each month on each path should match panel mean"

fig, axes = plot_partc_year_span_validation(YEAR_SPAN_BATCH, max_paths_price=10)
plt.show()

# YEAR_SPAN_BATCH["validation_by_month"].nlargest(5, "abs_err_max")
